In [1]:
from src.data.datasets.KVSS import KVSSDataModule
import numpy as np
from torch import nn
import torch
from hydra import initialize, compose
from omegaconf import DictConfig, OmegaConf

In [2]:
with initialize(config_path="config"):
    cfg = compose(config_name='defaults.yaml')
    print(OmegaConf.to_yaml(cfg))


<ipython-input-2-4e614cc2700c>:1: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path="config"):


data:
  name: kvss
  root: /tf/01_code/mylittlecodes/SleepVST_baseline/data/kvss
  video_dir: /tf/00_data/#_2021_Sleep_Video/
  signal_dir: /tf/01_code/mylittlecodes/SleepVST_baseline/data/kvss
  motion_dir: /tf/01_code/mylittlecodes/SleepVST_baseline/data/motionfeatures
  label_dir: /tf/00_AIoT2/video_signal/#_2021_Sleep_Video/30sec_labels
  model: SleepVST_BW
  seq_len: 240
  step_size: 60
  d_model: 128
  motion_dim: 90
  split: train
  exceptions:
  - A2019-EM-01-0119
  - A2019-EM-01-0120
  - A2019-EM-01-0122
  - A2019-EM-01-0123
  - A2019-EM-01-0124
  - A2019-EM-01-0125
  - A2019-EM-01-0196
  - A2019-EM-01-0197
  - A2019-EM-01-0198
  - A2019-EM-01-0199
  - A2019-EM-01-0200
  - A2019-EM-01-0201
  - A2019-EM-01-0202
  - A2019-EM-01-0203
  - A2019-EM-01-0204
  - A2019-EM-01-0205
  - A2019-EM-01-0206
  - A2021-EM-01-0163
  batch_size: 1
  num_workers: 8
  shuffle: true
  pin_memory: true
  data_source: raw
model:
  name: SleepVST_BW
  checkpoint: /tf/01_code/mylittlecodes/SleepVST_bas

In [3]:
cfg.command

'transfer_to_video'

In [ ]:
test_loader = kvss_test_module.get_dataloader()

all_final_features = []
all_labels = []
all_subject_ids = []

for batch_idx, batch in enumerate(test_loader):
    x_bw = batch['x_bw'].cuda().float() # (B, N, 150)
    labels = batch['label'].cpu().numpy()
    subject_ids = batch['subject_id']
    B, N, _ = x_bw.shape
    T = min(N, batch['label'].shape[1])
    batch_final_features = np.zeros((B, T, 218), dtype=np.float32)  # (B, T, 128 + 90)
    batch_labels = np.zeros((B, T), dtype=np.int64)  # (B, T)
    # print(batch_labels.shape)
    for t in range(T):
        batch_labels[:, t] = labels[:, t]
    for i in range(B):
        all_final_features.append(batch_final_features[i])
        all_labels.append(batch_labels[i])
        all_subject_ids.append(subject_ids[i])
y_test = np.hstack(labels for labels in all_labels)  # (num_samples, seq_len)
print(f"features shape: {np.array(all_final_features).shape}")
print(f"Total samples collected: {len(all_labels)}")
print(f"Total subjects collected: {len(all_subject_ids)}")
a = list(zip(all_final_features, all_labels, all_subject_ids))
print(len(a))
for i, (features, labels, subject_id) in enumerate(a):
    if subject_id == "A2020-EM-01-0110":
        for epoch_idx in range(len(labels)):
            print(f"epoch {epoch_idx} label: {labels[epoch_idx]}")

features shape: (70,)
Total samples collected: 70
Total subjects collected: 70
70
epoch 0 label: 0
epoch 1 label: 0
epoch 2 label: 0
epoch 3 label: 0
epoch 4 label: 0
epoch 5 label: 0
epoch 6 label: 0
epoch 7 label: 0
epoch 8 label: 0
epoch 9 label: 0
epoch 10 label: 0
epoch 11 label: 0
epoch 12 label: 0
epoch 13 label: 0
epoch 14 label: 0
epoch 15 label: 0
epoch 16 label: 0
epoch 17 label: 0
epoch 18 label: 0
epoch 19 label: 0
epoch 20 label: 0
epoch 21 label: 0
epoch 22 label: 0
epoch 23 label: 0
epoch 24 label: 0
epoch 25 label: 0
epoch 26 label: 1
epoch 27 label: 0
epoch 28 label: 1
epoch 29 label: 0
epoch 30 label: 0
epoch 31 label: 1
epoch 32 label: 0
epoch 33 label: 0
epoch 34 label: 0
epoch 35 label: 0
epoch 36 label: 0
epoch 37 label: 0
epoch 38 label: 0
epoch 39 label: 0
epoch 40 label: 0
epoch 41 label: 0
epoch 42 label: 0
epoch 43 label: 1
epoch 44 label: 1
epoch 45 label: 1
epoch 46 label: 1
epoch 47 label: 1
epoch 48 label: 1
epoch 49 label: 1
epoch 50 label: 1
epoch 51 l

<ipython-input-19-804d910a6efd>:22: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  y_test = np.hstack(labels for labels in all_labels)  # (num_samples, seq_len)
<ipython-input-19-804d910a6efd>:23: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  print(f"features shape: {np.array(all_final_features).shape}")


In [ ]:
s = np.zeros((2, 5), dtype=np.int64)